# Cleaning Method for AEMO Data

This notebook steps through the cleaning methods needed to transform the raw AEMO data into a useable dataset to be imported into Microsoft Power BI. It is informed by findings from the 01_exploration notebook. The stable functions live in src/transform.py

It also calculates a number of features that will be used in the analysis

### Cleaning Decisions Summary
- renaming columns to more user friendly names
- explicitly casting data types
- standardising timezone to AEST no DST as per AEMO


                       - canonical column names + types
#                       - canonical timezone (NEM time, AEST no DST)
#                       - duplicates: drop, keeping last
#                       - missing intervals: flag, don't fill
#                       - spike threshold: $300/MWh (justify)
#                       - features to engineer (list)

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path().resolve().parent / "src"))

In [2]:
import pandas as pd
import matplotlib as plt
from config import PROCESSED_DIR
import seaborn as sns
import numpy as np

In [3]:
historical = pd.read_parquet(PROCESSED_DIR / "historical_price_demand.parquet")

The column names as provided by AEMO are not intuitive. They've been replaced here, and units of measurement are included in the header

In [4]:
from transform import rename_cols

historical = rename_cols(historical)
print(historical)


        State     Settlement Date  Demand (MW)  Price ($/MWh) PERIODTYPE
0        NSW1 2024-01-01 00:05:00      6574.92          57.98      TRADE
1        NSW1 2024-01-01 00:10:00      6651.09          70.27      TRADE
2        NSW1 2024-01-01 00:15:00      6538.96          57.98      TRADE
3        NSW1 2024-01-01 00:20:00      6497.99          54.95      TRADE
4        NSW1 2024-01-01 00:25:00      6404.55          54.95      TRADE
...       ...                 ...          ...            ...        ...
1052635  VIC1 2025-12-31 23:40:00      4242.66           8.95      TRADE
1052636  VIC1 2025-12-31 23:45:00      4218.78           8.95      TRADE
1052637  VIC1 2025-12-31 23:50:00      4201.52           8.95      TRADE
1052638  VIC1 2025-12-31 23:55:00      4162.44           1.01      TRADE
1052639  VIC1 2026-01-01 00:00:00      4136.51          64.46      TRADE

[1052640 rows x 5 columns]


From the exploration step, the "PERIODTYPE" column contains only one value, "Trade." It is not useful information and is removed. 

In [5]:
from transform import drop_redundant_cols
print(historical["PERIODTYPE"].unique())
historical = drop_redundant_cols(historical)
print(historical.columns)

<ArrowStringArray>
['TRADE']
Length: 1, dtype: str
Index(['State', 'Settlement Date', 'Demand (MW)', 'Price ($/MWh)'], dtype='str')


This step explicity converts data types so that numerical columns are floats, dates are datetime and the State is category

In [6]:
from transform import cast_types
print(historical.dtypes)
historical = cast_types(historical)
print(f"\n{historical.dtypes}")

State                         str
Settlement Date    datetime64[us]
Demand (MW)               float64
Price ($/MWh)             float64
dtype: object

State                    category
Settlement Date    datetime64[us]
Demand (MW)               float64
Price ($/MWh)             float64
dtype: object


AEMO Time is AEST without Daylight Saving. Pandas imports as US datetime, so this needs to be converted

In [7]:
from transform import standardise_timezone
print(historical["Settlement Date"].dtype)
historical = standardise_timezone(historical)
print(historical["Settlement Date"].dtype)


datetime64[us]
datetime64[us, Australia/Brisbane]
